# 🧬 Generative vs. Discriminative Classification: Naive Bayes vs. Logistic Regression

**Subtitle:** Machine Learning for Scientists, Topics 3–4 — which probabilities can you trust, and *why*? Both models from scratch, tested on real tumour-measurement data.

| | |
|---|---|
| **Hardware Target** | CPU only (no GPU needed), Google Colab free tier |
| **Core Frameworks** | NumPy, scikit-learn (dataset, reference checks, calibration wrappers), Matplotlib |
| **Dataset** | Wisconsin Diagnostic Breast Cancer (569 samples, 30 features), bundled with scikit-learn, so no download is needed |
| **Prerequisites** | Basic Python/NumPy, Bayes' rule, gradient descent, train/test splits |
| **Estimated Runtime** | ~1 minute |

> ⚠️ **Disclaimer:** For research and educational purposes only, not to be used for clinical diagnosis.

> 🧭 **New to classification?** Start with the beginner course **BiologyToML**, modules [08 (k-NN and naïve Bayes)](../../BiologyToML/notebooks/08_knn_naive_bayes.ipynb) and [09 (logistic regression)](../../BiologyToML/notebooks/09_logistic_regression.ipynb), which build both models step by step. This notebook is a compact deep dive that assumes you already know the basics: it verifies from-scratch code against scikit-learn and *tests* why Naïve Bayes is overconfident.

---

## 🎯 What You Will Learn

1. How a **generative** classifier (Gaussian Naïve Bayes, models $P(X\mid Y)$) and a **discriminative** one (logistic regression, models $P(Y\mid X)$) differ, by implementing both from scratch and **verifying them against scikit-learn**.
2. How to compare two classifiers **fairly**: one train/test split can mislead, so we repeat the comparison over 100 splits.
3. How to measure the *quality of probabilities*, not just labels: **Brier score, log-loss, expected calibration error (ECE)** and reliability diagrams that show **how many samples** each point is based on, with **bootstrap confidence intervals**.
4. How to **test an explanation** instead of asserting it: *why* is Naïve Bayes overconfident here? We run ablations, and the obvious explanation turns out to be wrong.
5. How to **repair** probabilities with Platt scaling and isotonic regression, and what such a repair does and does not fix.

## 📋 Table of Contents

0. Environment Setup
1. Real Data: Wisconsin Breast Cancer Diagnostic Dataset
2. Generative vs. Discriminative: Theory
3. Implementation From Scratch (and verification)
4. Results: Accuracy, Decision Boundaries, Calibration, and What Causes Overconfidence
5. Production Context & Limitations
6. Summary
7. Questions & Bonus Exercises

## Module 0 — Environment Setup

In [ ]:
# ── Hardware Check ──
import sys, platform
print(f"Python {sys.version.split()[0]} | {platform.system()}")
print("This tutorial needs no GPU. CPU mode is fine ✅")

In [ ]:
# ── Install Dependencies ──
# Colab already ships these; the line is here so the notebook is self-contained.
!pip install -q numpy scikit-learn matplotlib

In [ ]:
# ── Global Imports & Configuration ──
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA

SEED = 0
np.random.seed(SEED)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
C_NB, C_LR = "#f0883e", ACCENT          # one colour per model, used in every figure
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})
print("✅ Imports OK")

---
# Module 1 — Real Data: Wisconsin Breast Cancer Diagnostic Dataset

Each of the 569 samples is a **fine-needle-aspirate image of a breast mass**. Ten nuclear properties (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension) are computed per image. For each we get the **mean, standard error and "worst" (largest)** value, giving 30 features. The task: **malignant vs. benign**.

We label **malignant = 1** (the "positive", disease class), so probabilities below are $P(\text{malignant}\mid x)$ and "false negative" means a missed cancer.

**Protocol (fixed from the start):**

1. One stratified **70/30 split**. The scaler (mean/std) is computed on the training part only.
2. No hyperparameter is tuned on the test set: logistic regression uses the fixed penalty `l2 = 1`.
3. In Module 4 we *also* repeat the analysis over 100 random splits. That is only to measure how much results vary from split to split. Nothing is selected using it.

In [ ]:
# ── Load Data, Split, Standardise (train statistics only) ──
data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)        # 1 = malignant (positive), 0 = benign
feature_names = list(data.feature_names)
print(f"Real dataset: {X.shape[0]} patients, {X.shape[1]} features computed from "
      f"digitised fine-needle-aspirate nucleus images")
print(f"Class balance: {(y == 1).sum()} malignant, {(y == 0).sum()} benign")

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
mu, sigma = Xtr.mean(0), Xtr.std(0)
Xtr_s, Xte_s = (Xtr - mu) / sigma, (Xte - mu) / sigma
print(f"Train: {len(ytr)} | Test: {len(yte)}")

In [ ]:
# ── Feature Correlations (training set): the independence assumption under stress ──
corr = np.corrcoef(Xtr_s.T)
i_rad, i_per, i_area = (feature_names.index(n) for n in ["mean radius", "mean perimeter", "mean area"])
print(f"Correlation(mean radius, mean perimeter) = {corr[i_rad, i_per]:.3f}")
print(f"Correlation(mean radius, mean area)      = {corr[i_rad, i_area]:.3f}")
iu = np.triu_indices_from(corr, k=1)
n_high = int((np.abs(corr[iu]) > 0.9).sum())
print(f"Feature pairs with |r| > 0.9: {n_high} of {len(iu[0])}")
print("Radius, perimeter and area are the same underlying tumour size measured three ways, "
      "so naive independence is clearly violated.")

fig, ax = plt.subplots(figsize=(7.5, 6.5), constrained_layout=True)
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Module 1 — Feature Correlations (training set)", color=ACCENT, fontweight="bold", fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

---
# Module 2 — Generative vs. Discriminative: Theory

## 2.1 Naïve Bayes (generative)

$$P(Y\mid X) \propto P(Y)\prod_{j=1}^{d} P(X_j \mid Y)$$

For continuous features, **Gaussian Naïve Bayes** models each $P(X_j\mid Y=c)$ as a 1-D Gaussian with its own mean and variance per class and feature. No covariance between features is modelled. Fitting is a single closed-form pass: per-class means, variances and class priors.

## 2.2 Logistic regression (discriminative)

$$P(Y=1\mid x) = \sigma(w_0 + w^\top x), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

There is no closed-form solution, so we fit by **gradient descent** on the negative log-likelihood. An $L_2$ penalty on $w$ is MAP estimation under a Gaussian prior (the same idea as ridge regression). In the code below the penalty strength is `l2`, which corresponds to scikit-learn's `C = 1/l2`.

## 2.3 A hypothesis worth testing

Naïve Bayes multiplies one likelihood per feature as if each were *new* evidence. When features are redundant, the same evidence is counted several times, which can push the predicted probability towards 0 or 1 (**overconfidence**) even while the *decision* (the arg max) stays right. This is why a classifier can have good accuracy and poor probabilities (Domingos & Pazzani, 1997). Logistic regression fits all weights jointly, so it can down-weight redundant features.

It is tempting to blame the three obvious size features (radius, perimeter, area). **We will test that explanation in Module 4.5 rather than assume it.**

---
# Module 3 — Implementation From Scratch

Both models in plain NumPy, then a check that they agree with scikit-learn. A from-scratch implementation is only trustworthy if it matches a reference.

In [ ]:
# ── Gaussian Naive Bayes and Logistic Regression, From Scratch ──
class GaussianNaiveBayes:
    """Per-class, per-feature Gaussian likelihood; no feature covariance modelled."""

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors, self.mean, self.var = {}, {}, {}
        for c in self.classes:
            Xc = X[y == c]
            self.priors[c] = len(Xc) / len(X)
            self.mean[c] = Xc.mean(0)
            self.var[c] = Xc.var(0) + 1e-6          # small variance floor (avoids division by zero)
        return self

    def _log_likelihood(self, X, c):
        return (-0.5 * np.sum(np.log(2 * np.pi * self.var[c]))
                - 0.5 * np.sum((X - self.mean[c]) ** 2 / self.var[c], axis=1))

    def predict_proba(self, X):
        logp = np.stack([self._log_likelihood(X, c) + np.log(self.priors[c]) for c in self.classes], axis=1)
        logp -= logp.max(axis=1, keepdims=True)      # log-sum-exp trick for numerical stability
        p = np.exp(logp)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes[np.argmax(self.predict_proba(X), axis=1)]


def sigmoid(z):
    """Numerically stable logistic function (never overflows for large |z|)."""
    e = np.exp(-np.abs(z))
    return np.where(z >= 0, 1 / (1 + e), e / (1 + e))


class LogisticRegressionScratch:
    """Gradient-descent logistic regression with an L2 (Gaussian-prior / MAP) penalty.
    The intercept is not penalised. l2 corresponds to scikit-learn's C = 1 / l2."""

    def __init__(self, lr=0.5, n_iters=3000, l2=1.0):
        self.lr, self.n_iters, self.l2 = lr, n_iters, l2

    def fit(self, X, y):
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        self.loss_history = []
        for _ in range(self.n_iters):
            p = sigmoid(X @ self.w + self.b)
            grad_w = X.T @ (p - y) / n + self.l2 * self.w / n
            grad_b = (p - y).mean()
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b
            eps = 1e-9
            self.loss_history.append(-np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)))
        return self

    def predict_proba(self, X):
        p1 = sigmoid(X @ self.w + self.b)
        return np.stack([1 - p1, p1], axis=1)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)


nb_model = GaussianNaiveBayes().fit(Xtr_s, ytr)
lr_model = LogisticRegressionScratch(lr=0.5, n_iters=3000, l2=1.0).fit(Xtr_s, ytr)

p_nb, p_lr = nb_model.predict_proba(Xte_s)[:, 1], lr_model.predict_proba(Xte_s)[:, 1]   # P(malignant)
pred_nb, pred_lr = nb_model.predict(Xte_s), lr_model.predict(Xte_s)
acc_nb, acc_lr = (pred_nb == yte).mean(), (pred_lr == yte).mean()
print(f"Test accuracy (this one split) — Naive Bayes: {acc_nb:.4f} | Logistic Regression: {acc_lr:.4f}")

fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
ax.plot(lr_model.loss_history, color=C_LR)
ax.set_xlabel("gradient descent iteration"); ax.set_ylabel("training negative log-likelihood")
ax.set_title("Module 3 — Logistic Regression Training Curve (no closed-form solution)", color=ACCENT, fontweight="bold")
plt.show()

In [ ]:
# ── Verify the Scratch Models Against scikit-learn ──
sk_nb = GaussianNB().fit(Xtr_s, ytr)
sk_lr = LogisticRegression(C=1.0 / lr_model.l2, max_iter=10000).fit(Xtr_s, ytr)          # default stopping tolerance
sk_lr_tight = LogisticRegression(C=1.0 / lr_model.l2, tol=1e-12, max_iter=100000).fit(Xtr_s, ytr)   # reference optimum
lr_long = LogisticRegressionScratch(lr=0.5, n_iters=30000, l2=1.0).fit(Xtr_s, ytr)       # same model, 10x more steps
p_ref = sk_lr_tight.predict_proba(Xte_s)[:, 1]

print("Gaussian NB  — predictions identical :", np.array_equal(sk_nb.predict(Xte_s), pred_nb))
print(f"               max |P_scratch - P_sklearn| = {np.abs(sk_nb.predict_proba(Xte_s)[:, 1] - p_nb).max():.2e}")
print("Logistic Reg — predictions identical :", np.array_equal(sk_lr.predict(Xte_s), pred_lr))
print(f"               scratch (3000 steps) vs sklearn default : max |ΔP| = {np.abs(sk_lr.predict_proba(Xte_s)[:, 1] - p_lr).max():.4f}")
print(f"               sklearn default      vs sklearn tight   : max |ΔP| = {np.abs(sk_lr.predict_proba(Xte_s)[:, 1] - p_ref).max():.4f}")
print(f"               scratch (3000 steps) vs sklearn tight   : max |ΔP| = {np.abs(p_lr - p_ref).max():.4f}")
print(f"               scratch (30000 steps) vs sklearn tight  : max |ΔP| = {np.abs(lr_long.predict_proba(Xte_s)[:, 1] - p_ref).max():.1e}")
print("\nThe small gap to default scikit-learn is optimiser stopping, not a bug: scikit-learn's default tolerance "
      "stops slightly short of the optimum, as does our fixed 3000 steps. Given enough steps, the scratch model "
      "reaches the same solution as a tightly converged scikit-learn.")

---
# Module 4 — Results: Accuracy, Decision Boundaries, Calibration, and What Causes Overconfidence

## 4.1 Beyond one accuracy number: the confusion matrix

Accuracy hides *which* mistakes are made. For screening, a **false negative** (a missed malignant case) is usually costlier than a false positive. We show the confusion matrices on the held-out test set, with sensitivity (malignant cases caught) and specificity (benign cases cleared).

In [ ]:
# ── Confusion Matrices on the Test Set ──
def confusion(y_true, y_pred):
    tn = int(((y_true == 0) & (y_pred == 0)).sum()); fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum()); tp = int(((y_true == 1) & (y_pred == 1)).sum())
    return np.array([[tn, fp], [fn, tp]])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), constrained_layout=True)
for ax, name, pred, col in [(axes[0], "Naive Bayes", pred_nb, C_NB), (axes[1], "Logistic Regression", pred_lr, C_LR)]:
    cm = confusion(yte, pred)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() * 1.3)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=16, fontweight="bold", color="black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["benign", "malignant"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["benign", "malignant"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    sens, spec = cm[1, 1] / cm[1].sum(), cm[0, 0] / cm[0].sum()
    ax.set_title(f"{name}\nsensitivity {sens:.3f} | specificity {spec:.3f} | missed cancers {cm[1, 0]}",
                 color=col, fontsize=10)
plt.show()
print(f"Test set size: {len(yte)} ({int(yte.sum())} malignant)")

## 4.2 Decision boundaries (two real features)

With only two features we can *see* what each model assumes. Naïve Bayes has a separate variance per class and feature, so its boundary is **curved** (quadratic). Logistic regression's boundary is **linear by construction**. Colour = predicted $P(\text{malignant})$. These two models are fitted on all samples purely for the picture; no accuracy is reported from them. Also look at the top-left corner, where there are almost no samples: Naïve Bayes's curved boundary bends back there, a reminder that a model's behaviour away from the data is set by its assumptions, not by evidence.

In [ ]:
# ── Decision Boundaries on Two Real Features ──
f1, f2 = "mean concave points", "mean texture"
i1, i2 = feature_names.index(f1), feature_names.index(f2)
X2 = X[:, [i1, i2]]
X2s = (X2 - X2.mean(0)) / X2.std(0)

nb2 = GaussianNaiveBayes().fit(X2s, y)
lr2 = LogisticRegressionScratch(lr=0.5, n_iters=3000, l2=1.0).fit(X2s, y)

xx, yy = np.meshgrid(np.linspace(X2s[:, 0].min() - 1, X2s[:, 0].max() + 1, 200),
                     np.linspace(X2s[:, 1].min() - 1, X2s[:, 1].max() + 1, 200))
grid = np.stack([xx.ravel(), yy.ravel()], axis=1)
nb_z = nb2.predict_proba(grid)[:, 1].reshape(xx.shape)
lr_z = lr2.predict_proba(grid)[:, 1].reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), constrained_layout=True)
fig.suptitle(f"Module 4.2 — Decision Boundaries ({f1} vs. {f2})", color=ACCENT, fontweight="bold")
for ax, z, title in zip(axes, [nb_z, lr_z], ["Naive Bayes (curved)", "Logistic Regression (linear)"]):
    ax.contourf(xx, yy, z, levels=20, cmap="RdBu_r", alpha=0.75)
    ax.scatter(X2s[y == 1, 0], X2s[y == 1, 1], s=10, c="darkred", label="malignant", edgecolor="k", linewidth=0.2)
    ax.scatter(X2s[y == 0, 0], X2s[y == 0, 1], s=10, c="darkblue", label="benign", edgecolor="k", linewidth=0.2)
    ax.set_title(title, color=TEXT, fontsize=10); ax.legend(fontsize=8)
    ax.set_xlabel(f1 + " (standardised)"); ax.set_ylabel(f2 + " (standardised)")
plt.show()

## 4.3 Is the accuracy gap real, or just this split?

The single split above gives one number per model. Different random splits give different numbers, so a gap seen on one split could be luck. We repeat the whole experiment on **100 random stratified 70/30 splits** (scaler and models always fitted on the training part only) and look at the *paired* difference in each split.

In [ ]:
# ── 100 Random Splits: Accuracy of Both Models ──
N_SPLITS = 100
acc_nb_s, acc_lr_s = [], []
for s in range(N_SPLITS):
    a, b, c, d = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
    m_, s_ = a.mean(0), a.std(0)
    a, b = (a - m_) / s_, (b - m_) / s_
    acc_nb_s.append((GaussianNaiveBayes().fit(a, c).predict(b) == d).mean())
    acc_lr_s.append((LogisticRegressionScratch(lr=0.5, n_iters=3000, l2=1.0).fit(a, c).predict(b) == d).mean())
acc_nb_s, acc_lr_s = np.array(acc_nb_s), np.array(acc_lr_s)
gap = acc_lr_s - acc_nb_s

print(f"Naive Bayes         accuracy over {N_SPLITS} splits: mean {acc_nb_s.mean():.3f} ± {acc_nb_s.std():.3f} "
      f"(range {acc_nb_s.min():.3f} to {acc_nb_s.max():.3f})")
print(f"Logistic Regression accuracy over {N_SPLITS} splits: mean {acc_lr_s.mean():.3f} ± {acc_lr_s.std():.3f} "
      f"(range {acc_lr_s.min():.3f} to {acc_lr_s.max():.3f})")
print(f"Paired difference (LR − NB): mean {gap.mean():+.3f}, median {np.median(gap):+.3f}, "
      f"min {gap.min():+.3f}, max {gap.max():+.3f}")
print(f"Logistic regression better in {(gap > 0).sum()} splits, tied in {(gap == 0).sum()}, worse in {(gap < 0).sum()}")
print(f"For comparison, the single split used in this notebook: NB {acc_nb:.3f}, LR {acc_lr:.3f} (gap {acc_lr - acc_nb:+.3f})")

fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
ax.hist(gap, bins=15, color=C_LR)
ax.axvline(0, color=TEXT, ls="--", lw=1)
ax.axvline(acc_lr - acc_nb, color=C_NB, lw=2, label="the single split used above")
ax.set_xlabel("accuracy(LR) − accuracy(NB) on the same split"); ax.set_ylabel("number of splits")
ax.set_title("Module 4.3 — Paired accuracy gap over 100 random splits", color=ACCENT, fontweight="bold")
ax.legend(facecolor=DARK_BG, labelcolor=TEXT)
plt.show()

**Reading the result.** Compare the spread of the accuracy within each model (±, range) with the average gap between them. A single test set of this size can move a model's accuracy by several percentage points, so one split should never be quoted as "the" accuracy. The *paired* view shows whether the ordering of the two models is stable. Note that this concerns **accuracy only**; the next sections show the models differ far more in the quality of their **probabilities**.

## 4.4 Calibration: are the probabilities believable?

A model is **calibrated** if, among samples it assigns probability $p$ of being malignant, a fraction $p$ truly are. We use three scores (lower is better for all):

* **Brier score**: mean squared error of the probability.
* **Log-loss**: punishes confident mistakes very heavily (a wrong answer given with probability 1 is catastrophic). We clip probabilities at $10^{-12}$.
* **Expected calibration error (ECE)**: average gap between predicted probability and observed frequency over 10 equal-width bins.

Because ~170 test samples is small, we also report **bootstrap 95% intervals** (2,000 resamples of the test set) for the *difference* between the models.

In [ ]:
# ── Calibration Metrics ──
def calibration_curve(y_true, y_prob, n_bins=10):
    """Mean predicted probability, observed fraction positive and sample count per equal-width bin."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_prob, bins) - 1, 0, n_bins - 1)
    mean_pred, frac_pos, counts = np.zeros(n_bins), np.zeros(n_bins), np.zeros(n_bins)
    for b in range(n_bins):
        mask = bin_ids == b
        counts[b] = mask.sum()
        if mask.sum() > 0:
            frac_pos[b], mean_pred[b] = y_true[mask].mean(), y_prob[mask].mean()
    return mean_pred, frac_pos, counts

def expected_calibration_error(mean_pred, frac_pos, counts):
    w, valid = counts / counts.sum(), counts > 0
    return np.sum(w[valid] * np.abs(mean_pred[valid] - frac_pos[valid]))

def metrics(y_true, p):
    """Brier, clipped log-loss, ECE and the fraction of 'extreme' predictions (P < 0.001 or P > 0.999)."""
    pc = np.clip(p, 1e-12, 1 - 1e-12)
    mp, fp, ct = calibration_curve(y_true, p)
    return {"Brier": np.mean((p - y_true) ** 2),
            "Log-loss": -np.mean(y_true * np.log(pc) + (1 - y_true) * np.log(1 - pc)),
            "ECE": expected_calibration_error(mp, fp, ct),
            "Extreme": ((p > 0.999) | (p < 0.001)).mean()}

def reliability_panel(ax, y_true, p, color, title):
    """Reliability diagram: dots only (no connecting lines); dot area is proportional to the samples in the bin."""
    mp, fp, ct = calibration_curve(y_true, p)
    ok = ct > 0
    ax.plot([0, 1], [0, 1], "--", color=TEXT, alpha=0.5)
    ax.scatter(mp[ok], fp[ok], s=25 + 10 * ct[ok], color=color, alpha=0.85, edgecolor="k", linewidth=0.3)
    for x_, y_, c_ in zip(mp[ok], fp[ok], ct[ok]):
        ax.annotate(str(int(c_)), (x_, y_), textcoords="offset points",
                    xytext=(-22, 8) if x_ > 0.5 else (8, 8), fontsize=8, color=color)
    ax.set_xlim(-0.08, 1.08); ax.set_ylim(-0.08, 1.15)
    ax.set_xlabel("mean predicted P(malignant)"); ax.set_ylabel("observed fraction malignant")
    ax.set_title(title, color=TEXT, fontsize=10)

m_nb, m_lr = metrics(yte, p_nb), metrics(yte, p_lr)
wrong_nb, wrong_lr = pred_nb != yte, pred_lr != yte
conf_nb = np.maximum(p_nb, 1 - p_nb)[wrong_nb]        # confidence the model had in its WRONG answers
conf_lr = np.maximum(p_lr, 1 - p_lr)[wrong_lr]

print(f"{'Metric (test set, one split)':<34}{'Naive Bayes':>13}{'Logistic Reg.':>15}")
print(f"{'Accuracy':<34}{acc_nb:>13.3f}{acc_lr:>15.3f}")
for k, label in [("Brier", "Brier score"), ("Log-loss", "Log-loss"), ("ECE", "Expected calibration error"),
                 ("Extreme", "Fraction of predictions at 0 or 1")]:
    print(f"{label:<34}{m_nb[k]:>13.3f}{m_lr[k]:>15.3f}")
print(f"{'Number of wrong predictions':<34}{int(wrong_nb.sum()):>13d}{int(wrong_lr.sum()):>15d}")
print(f"{'Median confidence when wrong':<34}{np.median(conf_nb):>13.3f}{np.median(conf_lr):>15.3f}")

# Bootstrap the DIFFERENCE (NB − LR) over test samples
rng = np.random.default_rng(SEED)
boot = {k: [] for k in ["Brier", "Log-loss", "ECE"]}
for _ in range(2000):
    i = rng.integers(0, len(yte), len(yte))
    a_, b_ = metrics(yte[i], p_nb[i]), metrics(yte[i], p_lr[i])
    for k in boot:
        boot[k].append(a_[k] - b_[k])
print("\nBootstrap 95% interval for the difference (Naive Bayes − Logistic Regression); positive = NB worse:")
for k, v in boot.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"  {k:<9} {m_nb[k] - m_lr[k]:+.3f}   [{lo:+.3f}, {hi:+.3f}]   "
          f"{'interval excludes 0' if lo > 0 or hi < 0 else 'interval includes 0'}")

In [ ]:
# ── Reliability Diagrams (with sample counts) and Probability Histograms ──
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), constrained_layout=True)
fig.suptitle("Module 4.4 — Reliability (numbers = samples in the bin; dot area ∝ samples) and prediction histograms",
             color=ACCENT, fontweight="bold", fontsize=11)
reliability_panel(axes[0, 0], yte, p_nb, C_NB, "Naive Bayes — reliability")
reliability_panel(axes[0, 1], yte, p_lr, C_LR, "Logistic Regression — reliability")
for ax, p, col, name in [(axes[1, 0], p_nb, C_NB, "Naive Bayes"), (axes[1, 1], p_lr, C_LR, "Logistic Regression")]:
    ax.hist(p, bins=30, color=col)
    ax.set_xlabel("predicted P(malignant)"); ax.set_ylabel("test samples")
    ax.set_title(f"{name}: {metrics(yte, p)['Extreme']:.0%} of predictions are below 0.001 or above 0.999",
                 color=TEXT, fontsize=10)
plt.show()
mp, fp, ct = calibration_curve(yte, p_nb)
print("Naive Bayes samples per bin (0–0.1, 0.1–0.2, …, 0.9–1.0):", ct.astype(int).tolist())
mp, fp, ct = calibration_curve(yte, p_lr)
print("Logistic Reg. samples per bin                          :", ct.astype(int).tolist())

**How to read this.**

* **Look at the bin counts.** When a model pushes almost every prediction to 0 or 1, the middle bins are empty or hold a single sample, so its reliability diagram is effectively *two dots*. ECE then measures little more than how far those two extreme bins are from 0 and 1. That is why we also use **log-loss**, which directly punishes the confident mistakes.
* **Confidence when wrong** is the cleanest symptom of overconfidence: a well-behaved model should be less sure when it errs.
* The **bootstrap intervals** tell you which differences are larger than resampling noise on this test set. Where an interval includes 0, do not claim a winner on that metric.

## 4.5 *Why* is Naïve Bayes overconfident here? An experiment, not an assumption

Hypothesis from Module 2.3: redundant features are counted repeatedly. The obvious suspects are **mean perimeter and mean area**, which are near-copies of mean radius. We test this and some alternatives. Every variant below is fitted on training data only, and the whole comparison is **averaged over 100 random splits** so a lucky split cannot decide the answer:

| Variant | What it tests |
|---|---|
| All 30 features | baseline |
| Drop mean perimeter and mean area | the "three size features" explanation |
| Top-*k* raw features (largest class separation on the training set) | fewer features, still correlated |
| PCA-whitened, *k* components | the same *number* of features, but **uncorrelated** with unit variance |
| PCA-whitened, all 30 | full decorrelation, no reduction |

If correlation among features drives overconfidence, then at the **same number of features** the decorrelated version should be better calibrated than the raw one. Metric means over the 100 splits: accuracy, log-loss, ECE, and the fraction of extreme predictions.

In [ ]:
# ── Ablation: What Reduces Naive Bayes Overconfidence? (100 splits, train-only fitting) ──
drop_idx = [feature_names.index(n) for n in ["mean perimeter", "mean area"]]
keep_idx = [j for j in range(X.shape[1]) if j not in drop_idx]

variants = ["All 30 features", "Drop mean perimeter + mean area",
            "Top-5 raw features", "PCA-whitened, 5 components",
            "Top-10 raw features", "PCA-whitened, 10 components",
            "PCA-whitened, all 30 components"]
rows = {v: [] for v in variants}

def evaluate(name, A, B, c, d):
    m = GaussianNaiveBayes().fit(A, c)
    p = m.predict_proba(B)[:, 1]
    mm = metrics(d, p)
    rows[name].append([(m.predict(B) == d).mean(), mm["Log-loss"], mm["ECE"], mm["Extreme"]])

for s in range(N_SPLITS):
    a, b, c, d = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
    m_, s_ = a.mean(0), a.std(0)
    a, b = (a - m_) / s_, (b - m_) / s_
    order = np.argsort(-np.abs(a[c == 1].mean(0) - a[c == 0].mean(0)))      # training-set class separation
    evaluate("All 30 features", a, b, c, d)
    evaluate("Drop mean perimeter + mean area", a[:, keep_idx], b[:, keep_idx], c, d)
    for k in (5, 10):
        evaluate(f"Top-{k} raw features", a[:, order[:k]], b[:, order[:k]], c, d)
        P = PCA(n_components=k, whiten=True).fit(a)
        evaluate(f"PCA-whitened, {k} components", P.transform(a), P.transform(b), c, d)
    P = PCA(n_components=a.shape[1], whiten=True).fit(a)
    evaluate("PCA-whitened, all 30 components", P.transform(a), P.transform(b), c, d)

abl = {v: np.array(r) for v, r in rows.items()}
print(f"{'Naive Bayes variant':<34}{'accuracy':>10}{'log-loss':>10}{'ECE':>8}{'extreme':>9}")
for v in variants:
    r = abl[v]
    print(f"{v:<34}{r[:, 0].mean():>10.3f}{r[:, 1].mean():>10.3f}{r[:, 2].mean():>8.3f}{r[:, 3].mean():>9.3f}")
print(f"\n(Means over {N_SPLITS} splits. 'extreme' = fraction of predictions below 0.001 or above 0.999.)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), constrained_layout=True)
ypos = np.arange(len(variants))[::-1]
for ax, col_i, title in [(axes[0], 1, "Log-loss (lower = better probabilities)"),
                         (axes[1], 3, "Fraction of extreme predictions (P<0.001 or P>0.999)")]:
    vals = [abl[v][:, col_i].mean() for v in variants]
    ax.barh(ypos, vals, color=[C_NB if "PCA" not in v else ACCENT for v in variants])
    ax.set_yticks(ypos); ax.set_yticklabels(variants, fontsize=8)
    ax.set_title(title, color=TEXT, fontsize=10)
fig.suptitle("Module 4.5 — orange = raw features, blue = PCA-whitened (decorrelated)", color=ACCENT, fontweight="bold")
plt.show()

**What the experiment shows** (compare the printed table; each statement is about the averages above):

1. **The "three size features" explanation does not hold.** Dropping mean perimeter and mean area leaves log-loss and the fraction of extreme predictions essentially unchanged. Many other features in this dataset are also strongly correlated (mean, standard-error and "worst" versions of the same measurements, and the concavity family), so removing two of them barely changes the total amount of repeated evidence.
2. **Using fewer features helps**, since there are fewer terms to over-count. The top-5 raw features have a much lower log-loss than all 30, yet most of their predictions are still extreme.
3. **At the same number of features, decorrelating helps much more on extremes.** PCA-whitened components give far fewer extreme predictions than the same number of raw features, and a lower log-loss at 10 features. This points to **correlation between features**, not any particular named feature, as the mechanism.
4. **Accuracy changes by only a couple of percentage points at most** across all variants except full whitening (compare the first column), while log-loss and the fraction of extreme predictions change a lot. This is consistent with the Domingos & Pazzani pattern: the *decision* survives, the *probabilities* do not.
5. **Decorrelation is not a free fix.** Whitening all 30 components is the *worst* variant on accuracy, log-loss and ECE. A likely reason is that the lowest-variance directions are poorly estimated from ~400 samples, and whitening inflates them to unit variance, so Naïve Bayes gives noise the same weight as signal.

**Caveats.** "Top-*k* raw features" is one way to pick a subset; other subsets could behave differently. PCA components are mixtures of original features, so they lose interpretability. And these are averages over splits of a single 569-sample dataset.

## 4.6 Repairing the probabilities: Platt scaling and isotonic regression

If the *ranking* is good but the probabilities are off, we can learn a small correction on top of the model. **Platt scaling** fits a sigmoid to the model's scores; **isotonic regression** fits a monotone step function (more flexible, needs more data). We use scikit-learn's `CalibratedClassifierCV` (5-fold, **fitted on the training set only**) around `GaussianNB`, which we verified above matches our scratch implementation. The test set is used once, for the final numbers.

In [ ]:
# ── Calibrating Naive Bayes (train-only), Evaluated Once on the Test Set ──
cal_sig = CalibratedClassifierCV(GaussianNB(), method="sigmoid", cv=5).fit(Xtr_s, ytr)
cal_iso = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(Xtr_s, ytr)
models = {"NB (raw)": p_nb,
          "NB + Platt (sigmoid)": cal_sig.predict_proba(Xte_s)[:, 1],
          "NB + isotonic": cal_iso.predict_proba(Xte_s)[:, 1],
          "Logistic Regression (raw)": p_lr}

print(f"{'Model (test set, one split)':<28}{'accuracy':>9}{'Brier':>8}{'log-loss':>10}{'ECE':>8}{'extreme':>9}")
for name, p in models.items():
    mm = metrics(yte, p)
    print(f"{name:<28}{((p > 0.5).astype(int) == yte).mean():>9.3f}{mm['Brier']:>8.3f}{mm['Log-loss']:>10.3f}"
          f"{mm['ECE']:>8.3f}{mm['Extreme']:>9.3f}")

fig, axes = plt.subplots(1, 4, figsize=(20, 4.8), constrained_layout=True, sharey=True)
fig.suptitle("Module 4.6 — Reliability before and after calibration (numbers = samples in the bin)",
             color=ACCENT, fontweight="bold", fontsize=11)
for ax, (name, p), col in zip(axes, models.items(), [C_NB, "#d2a8ff", "#7ee787", C_LR]):
    reliability_panel(ax, yte, p, col, name)
plt.show()

**Reading the table.** Compare each calibrated Naïve Bayes row with the raw one, and then with logistic regression. Both calibrators improve the probability metrics over raw Naïve Bayes. The decisions at the 0.5 threshold change little (check the accuracy column), because a monotone correction preserves the ranking of the samples and mostly rescales the confidence. Notice also that calibration is **not magic**: it repairs the *scale* of the probabilities, but if the ranking itself is worse than logistic regression's, a calibrated Naïve Bayes will not catch up on every metric. This is a single test split, so differences of a few thousandths should not be over-interpreted.

---
# Module 5 — Production Context & Limitations

| Method | Where used in practice | Notes |
|--------|------------------------|-------|
| Naïve Bayes | Spam and sequence classification, early gene-expression classifiers, text mining | Fast, needs little data. Its labels are often good, but its probabilities are unreliable when features are redundant (Module 4.5) |
| Logistic regression | GWAS, clinical risk scores, a baseline in many biomedical ML papers | Usually better-calibrated probabilities when the log-odds are roughly linear in the features; weights are directly interpretable |
| scikit-learn `GaussianNB` / `LogisticRegression` | Production implementations | Add a scaled variance floor, multiple solvers and regularisation paths (we matched them in Module 3) |
| Platt scaling / isotonic regression | Post-hoc calibration | A standard repair for an overconfident model; applied in Module 4.6 |

## Known Limitations of This Tutorial

* **One dataset, 569 samples.** Conclusions about *which* model wins, and about the overconfidence mechanism, are for this dataset. The mechanism (correlated features) should generalise; the sizes of the effects need not.
* **Calibration on one test split.** The test set has ~170 samples; use the bootstrap intervals and the repeated-split ablation, and see Bonus 3 for cross-validated calibration.
* **Equal-width ECE bins.** ECE is sensitive to binning, which is why we also report log-loss and show bin counts.
* **Diagonal-covariance Gaussian NB only.** "Semi-naïve" variants and full-covariance Gaussians (QDA) relax the independence assumption; not implemented here (Bonus 4).
* **Plain gradient descent with a fixed learning rate.** Production code uses L-BFGS or Newton-type solvers; our logistic regression matches scikit-learn only to optimiser tolerance.
* **Top-*k* feature subsets and PCA in Module 4.5** are one family of ablations among many.

In [ ]:
# ── Final Dashboard: the Whole Pipeline in One Figure ──
fig = plt.figure(figsize=(20, 10), constrained_layout=True)
fig.suptitle("Naive Bayes vs. Logistic Regression — Pipeline Dashboard", fontsize=15, fontweight="bold", color=ACCENT)
gs = fig.add_gridspec(2, 4)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax0.set_title("Feature correlations (train)", color=TEXT, fontsize=10); ax0.set_xticks([]); ax0.set_yticks([])
ax1 = fig.add_subplot(gs[0, 1]); ax1.contourf(xx, yy, nb_z, levels=20, cmap="RdBu_r", alpha=0.75)
ax1.scatter(X2s[:, 0], X2s[:, 1], c=y, cmap="RdBu_r", s=9, edgecolor="white", linewidth=0.3); ax1.set_title("Naive Bayes boundary (curved)", color=TEXT, fontsize=10)
ax1.set_xticks([]); ax1.set_yticks([])
ax2 = fig.add_subplot(gs[0, 2]); ax2.contourf(xx, yy, lr_z, levels=20, cmap="RdBu_r", alpha=0.75)
ax2.scatter(X2s[:, 0], X2s[:, 1], c=y, cmap="RdBu_r", s=9, edgecolor="white", linewidth=0.3); ax2.set_title("Logistic Regression boundary (linear)", color=TEXT, fontsize=10)
ax2.set_xticks([]); ax2.set_yticks([])
ax3 = fig.add_subplot(gs[0, 3]); ax3.plot([0, 1], [0, 1], "--", color=TEXT, alpha=0.5)
for p, col in [(p_nb, C_NB), (p_lr, C_LR)]:
    mp, fp, ct = calibration_curve(yte, p); ok = ct > 0
    ax3.scatter(mp[ok], fp[ok], s=15 + 8 * ct[ok], color=col, alpha=0.6)
ax3.set_title("Reliability: NB orange, LR blue (dot area ∝ samples)", color=TEXT, fontsize=10)

ax4 = fig.add_subplot(gs[1, 0:2])
names = ["Brier", "Log-loss", "ECE", "Extreme"]
xpos, w = np.arange(len(names)), 0.35
ax4.bar(xpos - w / 2, [m_nb[k] for k in names], w, label="Naive Bayes", color=C_NB)
ax4.bar(xpos + w / 2, [m_lr[k] for k in names], w, label="Logistic Regression", color=C_LR)
ax4.set_xticks(xpos); ax4.set_xticklabels(["Brier", "Log-loss", "ECE", "Fraction extreme"], fontsize=9)
ax4.set_title("Probability quality on the test set (lower = better for all four)", color=TEXT, fontsize=10)
ax4.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=8)
ax5 = fig.add_subplot(gs[1, 2]); ax5.hist(p_nb, bins=30, color=C_NB); ax5.set_title("NB predicted P(malignant)", color=TEXT, fontsize=9)
ax6 = fig.add_subplot(gs[1, 3]); ax6.hist(p_lr, bins=30, color=C_LR); ax6.set_title("LR predicted P(malignant)", color=TEXT, fontsize=9)
plt.show()

---
# Module 6 — Summary

| Question | What we found |
|---|---|
| **Are the from-scratch models right?** | Yes: Naïve Bayes matches scikit-learn's predictions and probabilities; logistic regression matches to optimiser tolerance (Module 3). |
| **Which has better accuracy?** | Logistic regression, consistently: it was better in nearly every one of the 100 splits (Module 4.3). Quote accuracy over many splits, not one, because the single-split gap is only one draw from a distribution. |
| **Which has better probabilities?** | Logistic regression: lower Brier score and log-loss, and far less confident when wrong. Use the bootstrap intervals to see which differences exceed noise (Module 4.4). |
| **Why is Naïve Bayes overconfident?** | Not because of radius/perimeter/area specifically. Dropping those barely changes anything. Redundancy among *many* features is the cause: at equal feature counts, decorrelated features give far fewer extreme predictions (Module 4.5). |
| **Can it be fixed?** | Substantially but not completely: Platt scaling or isotonic regression fitted on training data improves every probability metric, while accuracy changes little (Module 4.6). |

**Rules of thumb.** Never quote accuracy from a single split without its spread. If you use probabilities for decisions (risk scores, thresholds), measure log-loss and look at bin counts, not just accuracy or a reliability curve. State explanations as hypotheses until an ablation tests them.

In [ ]:
# ── Summary Table (computed from this run) ──
print(f"{'':<42}{'Naive Bayes':>13}{'Logistic Reg.':>15}")
print(f"{'Test accuracy, this split':<42}{acc_nb:>13.3f}{acc_lr:>15.3f}")
print(f"{'Test accuracy, mean over ' + str(N_SPLITS) + ' splits':<42}{acc_nb_s.mean():>13.3f}{acc_lr_s.mean():>15.3f}")
for k, label in [("Brier", "Brier score (lower = better)"), ("Log-loss", "Log-loss (lower = better)"),
                 ("ECE", "ECE (lower = better)")]:
    print(f"{label:<42}{m_nb[k]:>13.3f}{m_lr[k]:>15.3f}")
print(f"{'Median confidence in wrong predictions':<42}{np.median(conf_nb):>13.3f}{np.median(conf_lr):>15.3f}")

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Gaussian NB fitting | $\mathcal{O}(n \cdot d)$ | One pass over the data; closed form |
| Logistic regression fitting | $\mathcal{O}(\text{iters} \cdot n \cdot d)$ | Iterative gradient descent, no closed form |
| Prediction (either model) | $\mathcal{O}(n \cdot d)$ | Negligible |
| Calibration curve / ECE | $\mathcal{O}(n)$ | Negligible |
| Platt scaling | one extra 1-D logistic fit per fold | Negligible |

---
# Module 7 — Questions & Bonus Exercises

### 💬 Discussion Questions

**Q1.** In Module 4.3, how does the spread of each model's accuracy across splits compare with the average gap between the two models? What would you conclude if you had only looked at one split? Roughly how many test samples does one percentage point correspond to here?

**Q2.** Dropping mean perimeter and mean area did not reduce Naïve Bayes's overconfidence. Propose at least two reasons why, using the feature list (mean / standard error / "worst" versions of each measurement) and the correlation heatmap.

**Q3.** At the same number of features, PCA-whitened components are better calibrated than raw ones, yet whitening *all* 30 components is the worst variant. Why do low-variance directions hurt Naïve Bayes after whitening?

**Q4.** In a screening setting, a model that is wrong *with probability ≈ 1* is more dangerous than one that is wrong *with probability 0.7*. Which scores in Module 4.4 capture this, and which hide it? What does the bin-count printout tell you about the reliability diagram?

**Q5.** Platt scaling fits two parameters; isotonic regression fits a step function. With about 400 training samples, which would you expect to overfit more, and what evidence in the Module 4.6 table supports or contradicts that?

**Q6.** The Module 3 verification shows our logistic regression matching a tightly converged scikit-learn only with enough gradient steps. Change `n_iters` to 300 and then 30000. What happens to the loss curve, the agreement with scikit-learn and the calibration? Which is more sensitive to under-training, accuracy or probabilities?

**Q7.** Domingos & Pazzani (1997) argue Naïve Bayes can be an *optimal* classifier under zero-one loss even when independence is badly violated. Reconcile that with the overconfidence you measured: for what use-case does the violation matter, and for which does it not?

---

### 🚀 Bonus Exercises

**Bonus 1 — Duplicate features.** Replicate the whole standardised feature matrix $r \in \{1, 2, 4, 8, 16\}$ times and track accuracy and log-loss for both models. Which one changes, and why?

**Bonus 2 — Calibrate logistic regression too.** Wrap `LogisticRegression` in `CalibratedClassifierCV(method="sigmoid")` and add it to the Module 4.6 table. Does it help, hurt, or make no difference? Explain.

**Bonus 3 — Cross-validated calibration.** Replace the single test set by out-of-fold predictions on all 569 samples (`cross_val_predict(..., method="predict_proba")`) and recompute ECE and log-loss. How much do the intervals shrink?

**Bonus 4 — Full covariance.** Implement a Gaussian classifier with a full covariance matrix per class (QDA) or a shared one (LDA). Compare its calibration with Naïve Bayes and logistic regression. How does the number of parameters change with $d$?

**Bonus 5 — Better ECE.** Use equal-*count* (quantile) bins or 5 bins instead of 10. How do the ECE values and the ordering of the models change?

**Bonus 6 — Newton's method.** Implement logistic regression with Newton–Raphson (IRLS) and compare the iterations needed against gradient descent.

### 📚 References

* Domingos, P. & Pazzani, M. (1997). On the optimality of the simple Bayesian classifier under zero-one loss. *Machine Learning* 29, 103–130.
* Ng, A. Y. & Jordan, M. I. (2001). On discriminative vs. generative classifiers: a comparison of logistic regression and naive Bayes. *NIPS* 14.
* Niculescu-Mizil, A. & Caruana, R. (2005). Predicting good probabilities with supervised learning. *ICML*.
* Platt, J. (1999). Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. In *Advances in Large Margin Classifiers*.
* Street, W. N., Wolberg, W. H. & Mangasarian, O. L. (1993). Nuclear feature extraction for breast tumor diagnosis. *Proc. SPIE* 1905, 861–870 (the Wisconsin dataset).
* Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, Ch. 4 and 6.
* 02-620 *Machine Learning for Scientists* course materials, Lectures 3–4.